# 🔍 Attention Explorer: How Transformers Focus on Context

**Estimated Time:** 15–20 minutes

---

## What is Attention?

When predicting the next word, not all previous words matter equally.

Compare these two sentences:

- *"The **bank** approved my loan"* — financial institution
- *"The river **bank** was muddy"* — riverbank

**Attention** is the mechanism that lets a transformer decide *which tokens to focus on* when processing each position.

### Key Concepts:

Each token produces three vectors from its embedding:

| Vector | Intuition | Role |
|--------|-----------|------|
| **Query** (Q) | "What am I looking for?" | Compared against all Keys |
| **Key** (K) | "What do I offer?" | Matched against Queries |
| **Value** (V) | "What do I send if chosen?" | Weighted and summed |

The attention formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

- The result is a matrix where **each row sums to 1** — a probability distribution over past tokens
- The output for each token is a **weighted average of Values**
- GPT-2 is **causal**: each token can only attend to itself and *earlier* tokens

---

### How Q, K and V fit together

Here is the whole mechanism for one token. The word **its** produces a query, every word (including **its**) offers a key and a value, and the output is a weighted sum of the values:

![Query, Key and Value diagram](../diagrams/qkv/qkv.png)

1. Each word's embedding is projected into a **key** and a **value**; the word being processed also gets a **query**.
2. The query is compared with every key, then scaled by $\sqrt{d_k}$ and passed through softmax to give the **attention weights**.
3. Those weights decide how much of each **value** flows into the output. Here "dog" dominates, which helps predict *tail*.

The weights in the diagram are illustrative. The sections below measure the real ones from GPT-2.

---

## Setup: Importing Required Libraries

We will use:
- [transformers](https://huggingface.co/docs/transformers) — to load GPT-2 and extract its attention weights
- [torch](https://pytorch.org/) — for tensor operations
- [plotly](https://plotly.com/python/) — for interactive visualisations

In [1]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from transformers import AutoTokenizer, AutoModelForCausalLM, logging as hf_logging

hf_logging.set_verbosity_error()  # suppress download messages

/home/finley/Work/RDS/team_days/llm_team_day/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## Load Model

We use **GPT-2** — small enough to run on a laptop, but a real transformer with 12 attention layers and 12 heads per layer.

> `attn_implementation="eager"` loads the standard Python attention implementation, which lets us inspect and replace the attention weights directly.

In [2]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")
model.eval();

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4086.48it/s]


---

## Part 1: Visualising Attention Weights

GPT-2 produces an attention weight matrix for every layer and every head.
Each **row** of the matrix shows how much attention one token pays to every other token when it is being processed.

The helper below runs the model and returns the top next-token predictions alongside all attention tensors.

In [3]:
def predict_next(text, top_k=8):
    """Run GPT-2 on `text`; return top predictions and per-layer attention weights."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs, output_attentions=True)

    logits = out.logits[0, -1]                 # shape: (vocab_size,)
    probs  = torch.softmax(logits, dim=-1)
    top    = torch.topk(probs, top_k)

    predictions = [(tokenizer.decode([i]), v.item())
                   for i, v in zip(top.indices, top.values)]
    tok_labels  = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    return predictions, out.attentions, tok_labels

In [4]:
text = "The dog wagged its"

results, attentions, tok_labels = predict_next(text)

print(f'Prompt: "{text}"\n')
print("Top next-token predictions (with attention):")
for tok, score in results:
    bar = "█" * int(score * 60)
    print(f"  {repr(tok):15s} [{score:.3f}] {bar}")

Prompt: "The dog wagged its"

Top next-token predictions (with attention):
  ' tail'         [0.912] ██████████████████████████████████████████████████████
  ' head'         [0.014] 
  ' tails'        [0.009] 
  ' tongue'       [0.004] 
  ' fur'          [0.003] 
  ' neck'         [0.002] 
  ' hind'         [0.002] 
  ' nose'         [0.002] 


### Attention Heatmap

The heatmap below shows the attention weights from GPT-2's **last layer**, averaged across all 12 heads.

- **Rows** — the token doing the attending (Query)
- **Columns** — the token being attended to (Key)
- **Brighter** — more attention paid

Notice the blank upper-right triangle: GPT-2 is causal, so no token attends to tokens that come after it.

> Look at the last row (`its`) — which earlier token does it attend to most strongly?

In [5]:
# Last layer, averaged over heads → shape (seq_len, seq_len)
last_layer_attn = attentions[-1][0].mean(dim=0).numpy()

fig = go.Figure(go.Heatmap(
    z=last_layer_attn,
    x=tok_labels,
    y=tok_labels,
    colorscale="Blues",
    hovertemplate="%{y} → %{x}<br>weight: %{z:.3f}<extra></extra>",
    colorbar=dict(title="Attention<br>weight"),
))
fig.update_layout(
    title=(
        f"Attention Weights — Last Layer (avg over {model.config.n_head} heads)<br>"
        f'Prompt: "{text}"'
    ),
    xaxis_title="Attended to (Key)",
    yaxis_title="Attending from (Query)",
    xaxis=dict(tickangle=-30),
    width=650,
    height=580,
)
fig.show()

---

## Part 2: Next-Token Prediction — With vs Without Attention

What happens if we **remove** the selectivity of attention?

Instead of learned, weighted attention, we replace every attention matrix with **uniform weights** — every past token is attended to equally:

$$w_{ij} = \frac{1}{\text{number of past tokens}}$$

Without selective weighting, the model can no longer focus on relevant parts of the context — every word contributes equally regardless of meaning.

> We implement this by temporarily replacing the internal `_attn` method on each GPT-2 attention layer with a version that returns uniform weights.

In [6]:
from transformers import AttentionInterface
from transformers.masking_utils import AttentionMaskInterface, eager_mask


def uniform_attention_forward(module, query, key, value, attention_mask, **kwargs):
    """Replacement attention: every visible token gets the same weight."""
    b, h, sq, d = query.shape
    sk = key.shape[-2]
    # Uniform weights, but still respect the causal mask
    # so tokens cannot attend to positions that come after them.
    causal = torch.ones(sq, sk, device=query.device, dtype=torch.bool).tril(sk - sq)
    w = causal.to(query.dtype).expand(b, h, sq, sk)
    w = w / w.sum(dim=-1, keepdim=True).clamp(min=1e-9)  # normalise rows
    out = torch.matmul(w, value).transpose(1, 2)
    return out, w


AttentionInterface.register("uniform", uniform_attention_forward)
AttentionMaskInterface.register("uniform", eager_mask)


def use_uniform_attention(active: bool):
    """Toggle uniform (non-selective) attention across all GPT-2 layers."""
    model.config._attn_implementation = "uniform" if active else "eager"


In [7]:
def top_predictions(prompt, top_k=8):
    """Return top-k next-token predictions for `prompt` using the current model state."""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs)
    probs = torch.softmax(out.logits[0, -1], dim=-1)
    top   = torch.topk(probs, top_k)
    return [(tokenizer.decode([i]), v.item())
            for i, v in zip(top.indices, top.values)]


# ── With attention (normal) ───────────────────────────────────────────────────
preds_normal = top_predictions(text)

# ── Without attention (uniform weights) ──────────────────────────────────────
use_uniform_attention(True)
preds_uniform = top_predictions(text)
use_uniform_attention(False)   # always restore!


# ── Print comparison ──────────────────────────────────────────────────────────
print(f'Prompt: "{text}"\n')

print("With attention (learned, selective):")
for tok, score in preds_normal[:5]:
    bar = "█" * int(score * 60)
    print(f"  {repr(tok):15s} [{score:.3f}] {bar}")

print("\nWithout attention (uniform — every token weighted equally):")
for tok, score in preds_uniform[:5]:
    bar = "█" * int(score * 60)
    print(f"  {repr(tok):15s} [{score:.3f}] {bar}")

Prompt: "The dog wagged its"

With attention (learned, selective):
  ' tail'         [0.912] ██████████████████████████████████████████████████████
  ' head'         [0.014] 
  ' tails'        [0.009] 
  ' tongue'       [0.004] 
  ' fur'          [0.003] 

Without attention (uniform — every token weighted equally):
  ' w'            [0.041] ██
  ' a'            [0.022] █
  ' g'            [0.013] 
  ' the'          [0.010] 
  ' d'            [0.009] 


In [8]:
# Side-by-side bar chart comparing the two sets of predictions
n = 8
tok_n, sc_n = zip(*preds_normal[:n])
tok_u, sc_u = zip(*preds_uniform[:n])

fig = go.Figure([
    go.Bar(name="With attention",    x=list(tok_n), y=list(sc_n),
           marker_color="steelblue"),
    go.Bar(name="Uniform attention", x=list(tok_u), y=list(sc_u),
           marker_color="salmon"),
])
fig.update_layout(
    barmode="group",
    title=f'Next-Token Probabilities<br>Prompt: "{text}"',
    xaxis_title="Predicted token",
    yaxis_title="Probability",
    yaxis_range=[0, 1],
    legend=dict(x=0.55, y=0.95),
    width=750,
    height=430,
)
fig.show()

### Observation

With real attention, the model connects `"its"` back to `"dog"` (via coreference) and confidently predicts `" tail"`.

With uniform attention, that contextual signal is diluted equally across all tokens — the model can no longer anchor `"its"` to `"dog"`, and its confidence in `" tail"` drops noticeably.

---

## Exercise: Try Your Own Prompts

**Task:** Choose a sentence where context is important for predicting the next word, and compare the with- and without-attention outputs.

Good examples to try:
- `"She couldn't lift the box because it was too"` — does the model predict `" heavy"`?
- `"The bank by the river was"` — does attention help disambiguate *bank*?
- `"To make an omelette, you need to break some"`
- `"The capital of France is"`

**Questions to think about:**
1. Which word in the prompt is most important for the prediction?
2. Does the heatmap confirm that the model is attending to that word?
3. How much does the top prediction's probability drop without attention?

In [9]:
my_text = "She couldn't lift the box because it was too"  # ← change me!

_, my_attentions, my_tok_labels = predict_next(my_text)
preds_n = top_predictions(my_text)

use_uniform_attention(True)
preds_u = top_predictions(my_text)
use_uniform_attention(False)

print(f'Prompt: "{my_text}"\n')
print("With attention:")
for tok, score in preds_n[:5]:
    bar = "█" * int(score * 60)
    print(f"  {repr(tok):15s} [{score:.3f}] {bar}")
print("\nWithout attention:")
for tok, score in preds_u[:5]:
    bar = "█" * int(score * 60)
    print(f"  {repr(tok):15s} [{score:.3f}] {bar}")

Prompt: "She couldn't lift the box because it was too"

With attention:
  ' big'          [0.278] ████████████████
  ' heavy'        [0.252] ███████████████
  ' small'        [0.071] ████
  ' large'        [0.070] ████
  ' hard'         [0.027] █

Without attention:
  ' it'           [0.136] ████████
  ' the'          [0.058] ███
  ' much'         [0.039] ██
  ' and'          [0.038] ██
  ' a'            [0.033] █


In [10]:
# Attention heatmap for your sentence
my_last_layer = my_attentions[-1][0].mean(dim=0).numpy()

fig = go.Figure(go.Heatmap(
    z=my_last_layer,
    x=my_tok_labels,
    y=my_tok_labels,
    colorscale="Blues",
    hovertemplate="%{y} → %{x}<br>weight: %{z:.3f}<extra></extra>",
    colorbar=dict(title="Attention<br>weight"),
))
fig.update_layout(
    title=f'Attention Weights — Last Layer<br>"{my_text}"',
    xaxis_title="Attended to (Key)",
    yaxis_title="Attending from (Query)",
    xaxis=dict(tickangle=-30),
    width=700,
    height=600,
)
fig.show()

---

## 🎓 Key Takeaways

By now, you should understand:

✅ **What attention is** — a learned, selective weighting over past token Values  
✅ **Why it matters** — without it, all context is averaged equally and predictions degrade  
✅ **GPT-2 is causal** — each token attends only to itself and earlier tokens  
✅ **Multiple heads, multiple layers** — each head can specialise in different relationships (syntax, coreference, position, …)

### Practical Applications:

1. **Interpretability** — attention maps help diagnose what information a model relies on
2. **Long-context efficiency** — sparse or sliding-window attention reduces cost for long documents
3. **Prompt design** — understanding that context is selectively weighted helps you structure prompts so key information is salient

### 📚 Further Exploration:

- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — the best visual explanation of the attention mechanism
- [BertViz](https://github.com/jessevig/bertviz) — interactive per-head attention visualiser
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — the original 2017 paper

---

## ➕ Optional Extra: Attention From Scratch

> **This section is not required to follow the rest of the course.**  
> It walks through the core attention calculation step-by-step in plain NumPy.  
> Skip it if you like, or come back when you want to understand the maths!

The full scaled dot-product attention for one head is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

We can implement this in about ten lines of NumPy.

### Step 1: The Building Blocks

In [11]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))  # numerically stable
    return e / e.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, causal=True):
    """
    Q, K, V : (n_tokens, d_k)
    Returns  : output (n_tokens, d_v), weights (n_tokens, n_tokens)
    """
    d_k    = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)               # (n, n) similarity scores
    if causal:
        mask   = np.tril(np.ones_like(scores, dtype=bool))
        scores = np.where(mask, scores, -1e9)      # block future positions
    weights = softmax(scores)                      # (n, n) attention weights
    output  = weights @ V                          # (n, d_v) weighted sum
    return output, weights

### Step 2: Real vs Uniform Attention on a Toy Sequence

In [12]:
np.random.seed(7)
n_tokens, d_k = 5, 4

# In a real model these come from learned projection matrices:
#   Q = X @ W_Q,  K = X @ W_K,  V = X @ W_V
# Here we use random vectors to show the mechanics.
Q = np.random.randn(n_tokens, d_k)
K = np.random.randn(n_tokens, d_k)
V = np.random.randn(n_tokens, d_k)

_, weights_real = scaled_dot_product_attention(Q, K, V, causal=True)

# Uniform causal attention: token i attends equally to tokens 0 .. i
weights_uniform = np.zeros((n_tokens, n_tokens))
for i in range(n_tokens):
    weights_uniform[i, : i + 1] = 1.0 / (i + 1)

print("Real attention weights (learned, peaked):")
print(np.round(weights_real, 3))
print("\nUniform attention weights (flat average):")
print(np.round(weights_uniform, 3))

Real attention weights (learned, peaked):
[[1.    0.    0.    0.    0.   ]
 [0.011 0.989 0.    0.    0.   ]
 [0.469 0.178 0.353 0.    0.   ]
 [0.04  0.74  0.077 0.143 0.   ]
 [0.041 0.625 0.066 0.104 0.164]]

Uniform attention weights (flat average):
[[1.    0.    0.    0.    0.   ]
 [0.5   0.5   0.    0.    0.   ]
 [0.333 0.333 0.333 0.    0.   ]
 [0.25  0.25  0.25  0.25  0.   ]
 [0.2   0.2   0.2   0.2   0.2  ]]


### Step 3: Visualise Both Side-by-Side

In [13]:
labels = [f"t{i}" for i in range(n_tokens)]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Real attention (peaked)", "Uniform attention (flat)"],
    horizontal_spacing=0.15,
)

for col, (w, show_scale) in enumerate(
    [(weights_real, False), (weights_uniform, True)], start=1
):
    fig.add_trace(
        go.Heatmap(
            z=w, x=labels, y=labels,
            colorscale="Blues",
            zmin=0, zmax=1,
            showscale=show_scale,
            texttemplate="%{z:.2f}",
            textfont={"size": 11},
            hovertemplate="%{y} → %{x}: %{z:.3f}<extra></extra>",
            colorbar=dict(title="Weight"),
        ),
        row=1, col=col,
    )

fig.update_layout(
    title="Real vs Uniform Attention Weights (toy 5-token sequence)",
    width=750,
    height=390,
)
fig.show()

### 💡 Key Insight

- **Real attention** weights are *peaked* — each query focuses strongly on a small number of keys
- **Uniform attention** spreads weight evenly — the output is just a running average of the Value vectors
- The difference in output between the two quantifies exactly how much information the selective weighting provides

In a full transformer, this selectivity is learned over millions of training examples, allowing each head to specialise: one might track grammatical subjects, another might link pronouns to their referents, another might attend to positional neighbours.